# 🔬 mPES — Optimización Bayesiana en Colab Pro+ (GPU)

Ejecuta **optimización Bayesiana** (Optuna — TPE sampler) de los paquetes
**pes_dqn** (Deep Q-Network) o **pes_ac** (Advantage Actor-Critic) en
Google Colab Pro+ aprovechando la **GPU** para acelerar el entrenamiento.

| Aspecto | Detalle |
|---------|---------|
| **Objetivo** | Maximizar el **rendimiento normalizado medio** sobre las 64 secuencias de evaluación |
| **Enmascaramiento** | Acciones infactibles (`action > resources_left`) se suprimen antes del argmax |
| **Sampler** | TPE (Tree-structured Parzen Estimator), seed = 42 |
| **Pruner** | MedianPruner (`n_startup_trials=5`, `n_warmup_steps=2`) |
| **Almacenamiento** | SQLite — la DB persiste en Drive para reanudar tras desconexiones |

---

### 📋 Requisitos previos

1. Subir la carpeta `mPES/` a **Google Drive** (con `requirements.txt` en la raíz).
2. Seleccionar un runtime con **GPU** en Colab (`Runtime → Change runtime type → GPU`).

### 📁 Estructura en Drive

```
MyDrive/
├── mPES/                              ← repo source (código + inputs)
│   ├── requirements.txt
│   ├── pes_dqn/  pes_ac/  ...
└── mPES_results/                      ← outputs organizados
    └── {PACKAGE}/
        └── bayesian/
            └── {RESUME_DATE}/         ← DB Optuna + config + plots
```

### 🔄 Flujo de ejecución

| Paso | Celda | Descripción |
|------|-------|-------------|
| 0 | Mount Drive | Monta Google Drive en `/content/drive` |
| 1 | **Setup** | Copia el repo a `/content/mPES`, restaura la DB de Optuna, instala Python 3.12 + venv |
| 2 | **Optimización** | Lanza el subproceso con GPU, sincroniza la DB a Drive cada N minutos |
| 3 | **Resultados** | Muestra el ranking de trials y copia outputs finales a `mPES_results/` |

> **💡 Reconexión:** Si Colab se desconecta, reejecutar todas las celdas copia
> el repo, restaura la DB desde Drive y retoma automáticamente donde quedó.

### ⚙️ Notas sobre GPU y reproducibilidad

- La GPU se habilita con `CUDA_VISIBLE_DEVICES='0'` en el subproceso.
- El paquete configura automáticamente *memory growth* para evitar OOM.
- El seed (`SEED=42`) garantiza reproducibilidad **en CPU**.  En GPU, operaciones
  como `reduce_sum` pueden ser no-deterministas (orden de ejecución paralela).
- Para reproducibilidad estricta en GPU, agregar `TF_DETERMINISTIC_OPS=1` al
  entorno (con penalización de velocidad).

In [2]:
from google.colab import drive  # type: ignore[import-not-found]
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
#@title **Configuración** { display-mode: "form" }

PACKAGE = "pes_ac" #@param ["pes_dqn", "pes_ac"]
N_TRIALS = 100 #@param {type:"integer"}
RESUME_DATE = "2026-13-07" #@param {type:"string"}
DRIVE_REPO = "/content/drive/MyDrive/mPES" #@param {type:"string"}
DRIVE_RESULTS = "/content/drive/MyDrive/mPES_results" #@param {type:"string"}
DRIVE_SYNC_MINUTES = 15 #@param {type:"slider", min:1, max:30, step:1}
NTFY_TOPIC = "mpes-bayesian" #@param {type:"string"}
NTFY_EVERY_N = 5 #@param {type:"integer"}

# --- Derivados ---
import os

_MODULE_MAP = {'pes_dqn': 'optimize_dqn', 'pes_ac': 'optimize_ac'}
OPT_MODULE = _MODULE_MAP[PACKAGE]
REPO_DIR = '/content/mPES'

# --- Estructura de resultados en Drive ---
# mPES_results/{PACKAGE}/bayesian/{RESUME_DATE}/
DRIVE_OPT_DIR = os.path.join(DRIVE_RESULTS, PACKAGE, 'bayesian', RESUME_DATE)

DB_SUBDIR = f'{PACKAGE}/inputs/{RESUME_DATE}_BAYESIAN_OPT'
DB_NAME = f'optuna_study_{RESUME_DATE}.db'

# --- Python 3.12 venv (misma versión que linux_mpes_env) ---
PYTHON_VERSION = '3.12'
VENV_DIR = '/content/mpes_env'
PYTHON_BIN = os.path.join(VENV_DIR, 'bin', 'python')
PIP_BIN = os.path.join(VENV_DIR, 'bin', 'pip')

os.makedirs(DRIVE_OPT_DIR, exist_ok=True)
print(f'Paquete: {PACKAGE}  |  Módulo: {PACKAGE}.ext.{OPT_MODULE}')
print(f'Trials: {N_TRIALS}  |  Resume: {RESUME_DATE}')
print(f'Repo en Drive:      {DRIVE_REPO}')
print(f'Resultados en Drive: {DRIVE_OPT_DIR}')
print(f'DB: {DB_NAME}')
print(f'Python target: {PYTHON_VERSION}  |  Venv: {VENV_DIR}')

Paquete: pes_ac  |  Módulo: pes_ac.ext.optimize_ac
Trials: 100  |  Resume: 2026-13-07
Repo en Drive:      /content/drive/MyDrive/mPES
Resultados en Drive: /content/drive/MyDrive/mPES_results/pes_ac/bayesian/2026-13-07
DB: optuna_study_2026-13-07.db
Python target: 3.12  |  Venv: /content/mpes_env


## 1. 🛠️ Setup del entorno

Copia el repo desde Drive (si no existe localmente), restaura la DB de Optuna
(para reanudar estudios previos), instala **Python 3.12** en un venv aislado,
e instala todas las dependencias desde `requirements.txt` (mismas versiones
que `linux_mpes_env`).

> **Primera ejecución:** la instalación de Python 3.12 y dependencias toma ~3 min.
> **Ejecuciones posteriores:** usa el venv existente (~10 s).

In [ ]:
import shutil, subprocess, glob

# --- Copiar repo desde Drive (si no existe localmente) ---
if not os.path.exists(DRIVE_REPO):
    raise FileNotFoundError(
        f'No se encontró el repo en Drive: {DRIVE_REPO}\n'
        f'Subí la carpeta mPES a Google Drive antes de continuar.'
    )

if not os.path.exists(REPO_DIR):
    shutil.copytree(DRIVE_REPO, REPO_DIR)
    print(f'Repo copiado desde Drive ({DRIVE_REPO} → {REPO_DIR})')
else:
    print(f'Repo local existente: {REPO_DIR}')

# --- Limpiar __pycache__ (evita bytecode stale de otra máquina) ---
for cache_dir in glob.glob(os.path.join(REPO_DIR, '**', '__pycache__'), recursive=True):
    shutil.rmtree(cache_dir, ignore_errors=True)
print('__pycache__ limpiado')

# --- Garantizar __init__.py en sub-paquetes (Drive FUSE puede omitir archivos vacíos) ---
pkg_dir = os.path.join(REPO_DIR, PACKAGE)
for subpkg in ['config', 'ext', 'src']:
    init_path = os.path.join(pkg_dir, subpkg, '__init__.py')
    if not os.path.exists(init_path):
        os.makedirs(os.path.dirname(init_path), exist_ok=True)
        open(init_path, 'w').close()
        print(f'  Creado {PACKAGE}/{subpkg}/__init__.py (faltaba tras copiar desde Drive)')
    else:
        print(f'  OK  {PACKAGE}/{subpkg}/__init__.py')

# --- Restaurar SOLO la DB desde Drive ---
drive_db = os.path.join(DRIVE_OPT_DIR, DB_NAME)
repo_db = os.path.join(REPO_DIR, DB_SUBDIR, DB_NAME)
os.makedirs(os.path.dirname(repo_db), exist_ok=True)

if os.path.exists(drive_db):
    shutil.copy2(drive_db, repo_db)
    print(f'DB restaurada: {drive_db} → {repo_db}')
else:
    print(f'No se encontró DB previa en Drive ({drive_db}) — se iniciará desde cero.')

# --- Instalar Python 3.12 (misma versión que linux_mpes_env) ---
subprocess.run(
    ['add-apt-repository', '-y', 'ppa:deadsnakes/ppa'],
    check=True, capture_output=True
)
subprocess.run(['apt-get', 'update', '-qq'], check=True, capture_output=True)
subprocess.run([
    'apt-get', 'install', '-y', '-qq',
    f'python{PYTHON_VERSION}', f'python{PYTHON_VERSION}-venv',
    f'python{PYTHON_VERSION}-dev'
], check=True, capture_output=True)
print(f'Python {PYTHON_VERSION} instalado')

# --- Crear venv con Python 3.12 ---
if not os.path.exists(VENV_DIR):
    subprocess.run(
        [f'python{PYTHON_VERSION}', '-m', 'venv', VENV_DIR],
        check=True
    )
    print(f'Venv creado: {VENV_DIR}')
else:
    print(f'Venv existente: {VENV_DIR}')

# --- Instalar dependencias (mismas versiones que linux_mpes_env) ---
req_file = os.path.join(REPO_DIR, 'requirements.txt')
if not os.path.exists(req_file):
    raise FileNotFoundError(f'No se encontró {req_file}')

subprocess.run([PIP_BIN, 'install', '-q', '--upgrade', 'pip'], check=True)
subprocess.run([PIP_BIN, 'install', '-q', '-r', req_file], check=True)
print(f'Dependencias instaladas desde {req_file}')

# --- Verificar versiones y GPU ---
result = subprocess.run([PYTHON_BIN, '-c',
    'import sys, numpy, tensorflow as tf, optuna; '
    'gpus = tf.config.list_physical_devices("GPU"); '
    'print(f"Python {sys.version.split()[0]}, numpy {numpy.__version__}, '
    'TF {tf.__version__}, Optuna {optuna.__version__}"); '
    'gpu_name = gpus[0].name if gpus else "ninguna (CPU)"; '
    'print(f"GPU: {gpu_name}"); '
    '[tf.config.experimental.set_memory_growth(g, True) for g in gpus]; '
    'print("Memory growth: configurado") if gpus else None'
], capture_output=True, text=True)

if result.returncode != 0:
    print('ERROR en verificación del entorno:')
    print(result.stderr.strip())
    raise RuntimeError(f'Verificación falló (exit {result.returncode}). Revisar output arriba.')

print(result.stdout.strip())

if 'ninguna' in result.stdout:
    print('\n⚠️  No se detectó GPU. Verifica que el runtime de Colab sea GPU.')
    print('    Runtime → Change runtime type → GPU')
    print('    La optimización funcionará en CPU pero será más lenta.')
else:
    print('✅ GPU detectada — el entrenamiento usará aceleración por hardware.')

Repo copiado desde Drive (/content/drive/MyDrive/mPES → /content/mPES)
__pycache__ limpiado
  OK  pes_ac/config/__init__.py
  OK  pes_ac/ext/__init__.py
  OK  pes_ac/src/__init__.py
No se encontró DB previa en Drive (/content/drive/MyDrive/mPES_results/pes_ac/bayesian/2026-13-07/optuna_study_2026-13-07.db) — se iniciará desde cero.


In [6]:
import subprocess, os, sqlite3

subprocess.run(['nvidia-smi'], check=False)

with open('/proc/meminfo') as f:
    for line in f:
        if 'MemTotal' in line:
            print(f'\nRAM total: {int(line.split()[1]) / 1024**2:.1f} GB')
            break

# --- Verificar archivos ---
repo_db = os.path.join(REPO_DIR, DB_SUBDIR, DB_NAME)
csv1 = os.path.join(REPO_DIR, PACKAGE, 'inputs', 'sequence_lengths.csv')
csv2 = os.path.join(REPO_DIR, PACKAGE, 'inputs', 'initial_severity.csv')

print('\nArchivos requeridos:')
for fpath in [repo_db, csv1, csv2]:
    ok = os.path.exists(fpath)
    print(f'  {"OK" if ok else "FALTA"}  {os.path.relpath(fpath, REPO_DIR)}')

# --- Estado de la DB ---
if os.path.exists(repo_db):
    conn = sqlite3.connect(repo_db)
    has_trials = conn.execute(
        "SELECT COUNT(*) FROM sqlite_master WHERE type='table' AND name='trials'"
    ).fetchone()[0]
    if has_trials:
        print('\nEstado de trials:')
        for row in conn.execute('SELECT state, COUNT(*) FROM trials GROUP BY state'):
            print(f'  {row[0]}: {row[1]}')
        best = conn.execute(
            'SELECT MAX(tv.value) FROM trial_values tv '
            'JOIN trials t ON tv.trial_id=t.trial_id WHERE t.state="COMPLETE"'
        ).fetchone()
        if best and best[0]:
            print(f'  Mejor valor: {best[0]:.6f}')
    else:
        print('\nDB existe pero está vacía (no hay estudios previos) — se iniciará desde cero.')
    conn.close()
else:
    print('\nNo hay DB previa — se creará al iniciar la optimización.')


RAM total: 51.0 GB

Archivos requeridos:
  OK  pes_dqn/inputs/2026-13-07_BAYESIAN_OPT/optuna_study_2026-13-07.db
  OK  pes_dqn/inputs/sequence_lengths.csv
  OK  pes_dqn/inputs/initial_severity.csv

Estado de trials:
  FAIL: 2


## 2. 🚀 Ejecutar optimización (GPU)

Lanza la optimización como subproceso usando el venv Python 3.12 **con GPU habilitada**.
La función objetivo evalúa cada trial en las 64 secuencias fijas y devuelve el
**rendimiento normalizado medio** (`mean_perf`).

| Funcionalidad | Detalle |
|---------------|---------|
| **Sync a Drive** | Un hilo respalda la DB cada `DRIVE_SYNC_MINUTES` minutos |
| **Notificaciones** | Push vía [ntfy.sh](https://ntfy.sh) cada N trials completados |
| **Reconexión** | Si Colab se desconecta, reejecutar restaura la DB y retoma |

> **⏱️ Duración típica:** ~2–5 min por trial (DQN) o ~3–8 min por trial (A2C),
> dependiendo de `num_episodes` y la GPU asignada.

In [ ]:
import subprocess, os, threading, time, shutil, re, sqlite3
import urllib.request

db_src = os.path.join(REPO_DIR, DB_SUBDIR, DB_NAME)
db_dst = os.path.join(DRIVE_OPT_DIR, DB_NAME)


def ntfy(msg, title=None, priority='default', tags=''):
    """Envía notificación push vía ntfy.sh."""
    if not NTFY_TOPIC:
        return
    try:
        headers = {}
        if title:
            safe = title.encode('latin-1', errors='ignore').decode('latin-1').strip()
            if safe:
                headers['Title'] = safe
        if priority and priority != 'default':
            headers['Priority'] = priority
        if tags:
            headers['Tags'] = tags
        req = urllib.request.Request(
            f'https://ntfy.sh/{NTFY_TOPIC}',
            data=msg.encode('utf-8'), headers=headers, method='POST'
        )
        urllib.request.urlopen(req, timeout=10)
    except Exception as e:
        print(f'  [ntfy] Error: {e}', flush=True)


def query_db(db_path):
    """Retorna (n_complete, best_value) desde la DB de Optuna."""
    try:
        conn = sqlite3.connect(db_path)
        n = conn.execute("SELECT COUNT(*) FROM trials WHERE state='COMPLETE'").fetchone()[0]
        best = conn.execute(
            "SELECT MAX(tv.value) FROM trial_values tv "
            "JOIN trials t ON tv.trial_id=t.trial_id WHERE t.state='COMPLETE'"
        ).fetchone()[0]
        conn.close()
        return n, best
    except Exception:
        return 0, None


def fmt_dur(s):
    """Segundos a formato legible."""
    if s < 60: return f'{s:.0f}s'
    if s < 3600: return f'{int(s//60)}m {int(s%60)}s'
    return f'{int(s//3600)}h {int((s%3600)//60)}m'


def fmt_best(val):
    """Formatea el mejor valor, o '---' si es None."""
    return f'{val:.6f}' if val is not None else '---'


# --- Hilo de sync a Drive ---
done_event = threading.Event()

def sync_to_drive():
    while not done_event.is_set():
        done_event.wait(DRIVE_SYNC_MINUTES * 60)
        if done_event.is_set():
            break
        try:
            shutil.copy2(db_src, db_dst)
            print(f'  [Drive Sync {time.strftime("%H:%M:%S")}] DB respaldada en {DRIVE_OPT_DIR}', flush=True)
        except Exception as e:
            print(f'  [Drive Sync] Error: {e}', flush=True)

threading.Thread(target=sync_to_drive, daemon=True).start()

# --- Entorno y comando (usa el venv Python 3.12) ---
env = os.environ.copy()
env.update({
    'CUDA_VISIBLE_DEVICES': '0',
    'VIRTUAL_ENV': VENV_DIR,
    'PATH': os.path.join(VENV_DIR, 'bin') + ':' + env.get('PATH', ''),
    'PYTHONPATH': REPO_DIR,
    'PYTHONUNBUFFERED': '1',
    'PYTHONIOENCODING': 'utf-8',
    'TF_ENABLE_ONEDNN_OPTS': '0',
    'TF_CPP_MIN_LOG_LEVEL': '2',
})

# --- Pre-flight: verificar que el módulo completo se puede importar ---
full_module = f'{PACKAGE}.ext.{OPT_MODULE}'
preflight = subprocess.run(
    [PYTHON_BIN, '-c', f'import {full_module}; print("{full_module} OK")'],
    cwd=REPO_DIR, env=env, capture_output=True, text=True
)
if preflight.returncode != 0:
    print(f'ERROR: no se puede importar {full_module}:')
    print(preflight.stderr.strip())
    raise RuntimeError(
        f'Import de {full_module} falló (exit {preflight.returncode}). '
        f'Revisar output arriba.'
    )
print(preflight.stdout.strip())

cmd = [PYTHON_BIN, '-m', full_module, str(N_TRIALS), '--resume', RESUME_DATE]

# --- Estado inicial y notificación ---
initial_n, initial_best = query_db(db_src)
ntfy(
    f'Optimización iniciada\nPaquete: {PACKAGE}\n'
    f'Trials: {initial_n}/{N_TRIALS}\nMejor: {fmt_best(initial_best)}',
    title=f'{PACKAGE} — Iniciado', tags='rocket'
)

print(f'▶ {" ".join(cmd)}')
print(f'  cwd: {REPO_DIR}  |  Reanudando desde trial {initial_n}/{N_TRIALS}')
if initial_best:
    print(f'  Mejor valor actual: {initial_best:.6f}')
print()

# --- Ejecutar subproceso ---
completed = [initial_n]
last_notified = [initial_n]
best_value = initial_best
t_start = time.time()
_re_best = re.compile(r'[Bb]est\s+value[:\s]+([0-9]+\.[0-9]+)')

proc = subprocess.Popen(
    cmd, cwd=REPO_DIR, env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1
)

for line in proc.stdout:
    m = _re_best.search(line)
    if m:
        best_value = float(m.group(1))

    if re.search(r'[Ff]inished|COMPLETE|complete', line):
        n, val = query_db(db_src)
        if n > 0:
            completed[0] = n
        if val is not None:
            best_value = val

        # Notificación periódica
        if (completed[0] - last_notified[0]) >= NTFY_EVERY_N:
            last_notified[0] = completed[0]
            elapsed = time.time() - t_start
            eta = (elapsed / max(completed[0], 1)) * (N_TRIALS - completed[0])
            ntfy(
                f'Progreso: {completed[0]}/{N_TRIALS}\n'
                f'Mejor: {fmt_best(best_value)}\n'
                f'Tiempo: {fmt_dur(elapsed)}  |  ETA: {fmt_dur(eta)}',
                title=f'{PACKAGE} — {completed[0]}/{N_TRIALS}',
                tags='chart_with_upwards_trend'
            )

    print(line, end='', flush=True)

# --- Finalización ---
exit_code = proc.wait()
done_event.set()
elapsed_total = time.time() - t_start

final_n, final_best = query_db(db_src)
if final_n > 0: completed[0] = final_n
if final_best is not None: best_value = final_best

# Backup final
shutil.copy2(db_src, db_dst)

print(f'\n{"="*50}')
print(f'  FINALIZADO  |  Trials: {completed[0]}/{N_TRIALS}  |  '
      f'Mejor: {fmt_best(best_value)}  |  '
      f'Tiempo: {fmt_dur(elapsed_total)}  |  Exit: {exit_code}')
print(f'{"="*50}')

ntfy(
    f'Finalizado\nTrials: {completed[0]}/{N_TRIALS}\n'
    f'Mejor: {fmt_best(best_value)}\n'
    f'Tiempo: {fmt_dur(elapsed_total)}\n'
    f'Exit: {exit_code}',
    title=f'{PACKAGE} — Finalizado',
    priority='high',
    tags='white_check_mark' if exit_code == 0 else 'warning'
)

  EXPERIMENT CONFIGURATION PARAMETERS

Variable Name                                 Variable Value                 Suggested Value     
----------------------------------------------------------------------------------------------------
AVAILABLE_RESOURCES_PER_SEQUENCE              39                             49                  
INIT_NO_OF_CITIES                             2                             
NUM_BLOCKS                                    8                              8                   
NUM_MAX_TRIALS                                10                             10                  
NUM_MIN_TRIALS                                3                              3                   
NUM_SEQUENCES                                 8                              8                   
PANDEMIC_PARAMETER                            0.4                            0.6                 
PLAYER_TYPE                                   DQN_AGENT                     
RESPONSE_MULTIPLIER 

## 3. 📊 Resultados

Muestra el ranking de trials completados (Top 10) y copia **todos** los outputs
finales a Drive: `mPES_results/{PACKAGE}/bayesian/{RESUME_DATE}/`

Archivos copiados:
- `optuna_study_*.db` — base de datos del estudio (reanudable)
- `optimization_results_*.txt` — reporte con hiperparámetros y métricas
- `optimization_history_*.png` — gráfico de convergencia
- `hyperparameter_importances_*.png` — importancia de cada hiperparámetro
- `dqn_best_*.keras` / `ac_best_*.keras` — mejor modelo encontrado
- `rewards_best_*.npy` — historial de rewards del mejor entrenamiento

In [ ]:
import sqlite3, os, shutil

db = os.path.join(REPO_DIR, DB_SUBDIR, DB_NAME)
conn = sqlite3.connect(db)

print('Trials por estado:')
for row in conn.execute('SELECT state, COUNT(*) FROM trials GROUP BY state'):
    print(f'  {row[0]}: {row[1]}')

print('\nTop 10 trials:')
rows = conn.execute(
    'SELECT t.trial_id, tv.value '
    'FROM trial_values tv JOIN trials t ON tv.trial_id=t.trial_id '
    'WHERE t.state="COMPLETE" ORDER BY tv.value DESC LIMIT 10'
).fetchall()
for i, row in enumerate(rows, 1):
    marker = ' (BEST)' if i == 1 else ''
    print(f'  #{row[0]:3d}  {row[1]:.6f}{marker}')
conn.close()

# --- Copiar outputs completos a mPES_results/{PACKAGE}/bayesian/{DATE}/ ---
opt_dir_src = os.path.join(REPO_DIR, DB_SUBDIR)
if os.path.exists(opt_dir_src):
    shutil.copytree(opt_dir_src, DRIVE_OPT_DIR, dirs_exist_ok=True)
    print(f'\nOutputs copiados a: {DRIVE_OPT_DIR}')
    for fname in sorted(os.listdir(DRIVE_OPT_DIR)):
        size = os.path.getsize(os.path.join(DRIVE_OPT_DIR, fname))
        print(f'  {fname} ({size/1024:.0f} KB)')